In [1]:
library(tidyverse)
library(edgeR)
library(ggrastr)
library(gridExtra)
library(RColorBrewer)
library(patchwork)
library(pheatmap)
library(rtracklayer)
library(GenomicRanges)
theme_set(theme_bw() + theme(panel.grid=element_blank(), text=element_text(size=15)))

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Loading required package: limma


Attaching package: ‘gridExtra’


The following object is masked from ‘package:dplyr’:

    combine


Loading required package: GenomicRanges

Loading required package: stats4

Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following object is masked from ‘package:gridExtra’:

    combine


The following object is masked from ‘package:limma’:

    plotMA


The following objects are masked from ‘package:lub

In [2]:
#read in counts data
countmat_1h <- read.delim("/mnt/disk/home/agulka/manuscripts/BAF_stimulus_CellGenomics/inputs/countmat/GM12878_stimulus/GM_stim_1h_counts_1.tsv", header=F, sep='\t')
countmat_6h <- read.delim("/mnt/disk/home/agulka/manuscripts/BAF_stimulus_CellGenomics/inputs/countmat/GM12878_stimulus/GM_stim_6h_counts_1.tsv", header=F, sep='\t')

#read in row metadata
row_meta <- read.delim("/mnt/disk/home/agulka/manuscripts/BAF_stimulus_CellGenomics/inputs/bed/ATAC_peaks/filtered/consensus_filt_annot.tsv", sep='\t')

#read in column metadata
col_meta_1h <- read.delim("/mnt/disk/home/agulka/manuscripts/BAF_stimulus_CellGenomics/inputs/countmat/GM12878_stimulus/metadata.txt", sep='\t')
col_meta_6h <- col_meta_1h

#filter to only include counts (exclude genomic coordinates)
counts_1h <- countmat_1h[,5:16]
counts_6h <- countmat_6h[,5:16]

In [3]:
#read in control counts (reads over promoters of genes invariantly expressed in BAFinh 6h/10µM vs DMSO
control_mat_1h <- read.delim("/mnt/disk/home/agulka/manuscripts/BAF_stimulus_CellGenomics/inputs/countmat/GM12878_stimulus/GM_stim_1h_invariant_promoter_counts.tsv", header=F, sep='\t')
control_mat_6h <- read.delim("/mnt/disk/home/agulka/manuscripts/BAF_stimulus_CellGenomics/inputs/countmat/GM12878_stimulus/GM_stim_6h_invariant_promoter_counts.tsv", header=F, sep='\t')
#calculate effective library sizes for normalization (sum of counts over each control region set)
lib_sizes_1h <- colSums(control_mat_1h[,5:16])
lib_sizes_6h <- colSums(control_mat_6h[,5:16])

#add library sizes to column metadata
col_meta_1h <- col_meta_1h %>% mutate(sizes=lib_sizes_1h)
col_meta_6h <- col_meta_6h %>% mutate(sizes=lib_sizes_6h)

In [4]:
# Differential Accessibility Region (DAR) Analysis with Custom Normalization
#
# This function performs differential accessibility analysis using edgeR,
# incorporating custom normalization based on user-supplied effective library sizes 
# The specific use case is for normalizing ATAC-seq using counts from promoter regions of genes that are invariantly expressed in control and BAF-inhibited samples.
#
# Inputs are:
#     data,  A matrix of raw counts (peaks x samples).
#     coldata, A data frame containing sample metadata. Must include:
#      `Condition`: Factor or character vector indicating experimental groups (e.g., control, treatment).
#      `sizes`: Numeric vector of effective library sizes (i.e., sum of counts at invariant promoters)
#
# This function returns a data frame with differential test results for each region, including logFC, p-values, and FDR
# As well as a classification of each element as gaining ("up"), losing ("down"), or not changing ("nc") in accessibility
# Based on FDR and logFC cutoffs of 0.05 and log2(1.5), respectively

dar_standard <- function(data, coldata) {
    library(edgeR)
  # Create DGEList object with counts and group information
    dge <- DGEList(counts = data, group = coldata$Condition)

    dge <- calcNormFactors(dge)
  
  # Create design matrix for comparison (model without intercept, group-specific columns)
    design <- model.matrix(~0 + group, data = dge$samples)
    colnames(design) <- levels(dge$samples$group)  # optional: label columns by condition names
  
  # Estimate dispersion 
    dge <- estimateDisp(dge, design)
  
  # Fit generalized linear model (GLM) 
    dge_fit <- glmQLFit(dge, design)
  
  # Conduct differential test (treatment vs control)
    diff <- glmQLFTest(dge_fit, contrast = c(1, -1))
  
  # Return all regions without sorting; include logFC, p-value, FDR, etc.
    res <- topTags(diff, n = Inf, sort.by = "none")$table 
  # Add column categorizing peaks as gaining, losing, or not changing accessibility
    res <- res %>% mutate(
                    change = case_when(FDR < 0.05 & logFC > log2(1.5) ~ "up",
                                       FDR < 0.05 & logFC < -log2(1.5) ~ "down",
                                       TRUE ~ "ns"))
  return(res)
}

dar <- function(data, coldata) {
    library(edgeR)
  
  # Create DGEList object with counts and group information
    dge <- DGEList(counts = data, group = coldata$Condition)
  
  # Override default library size with effective library sizes from control regions
    dge$samples$lib.size <- coldata$sizes
  
  # Set normalization factors to 1 (i.e., use lib.size as-is with no further scaling)
    dge$samples$norm.factors <- rep(1, nrow(coldata))
  
  # Create design matrix for comparison (model without intercept, group-specific columns)
    design <- model.matrix(~0 + group, data = dge$samples)
    colnames(design) <- levels(dge$samples$group)  # optional: label columns by condition names
  
  # Estimate dispersion 
    dge <- estimateDisp(dge, design)
  
  # Fit generalized linear model (GLM) 
    dge_fit <- glmQLFit(dge, design)
  
  # Conduct differential test (treatment vs control)
    diff <- glmQLFTest(dge_fit, contrast = c(1, -1))
  
  # Return all regions without sorting; include logFC, p-value, FDR, etc.
    res <- topTags(diff, n = Inf, sort.by = "none")$table 
  # Add column categorizing peaks as gaining, losing, or not changing accessibility
    res <- res %>% mutate(
                    change = case_when(FDR < 0.05 & logFC > log2(2) ~ "up",
                                       FDR < 0.05 & logFC < -log2(2) ~ "down",
                                       TRUE ~ "ns"))
  return(res)
}

# This function implements the dar() function separately for TSS-distal and TSS-proximal peaks. 
# In addition to the data and coldata params required by dar(), this function additionall requires
# a 'rowdata' object (should contain minimally genomic coordinates and TSS proximity for each row)

# This function returns a list object where the first element is the table of results from DAR analysis, 
# And the second is a reformatted rowdata object (rows reordered so that they match the results table)

dar_splitTSS <- function(data, coldata, rowdata){ 
    #Use row metadata to split counts matrix into TSS-proximal and dista
    #Run DAR analysis separately on each
    res_tss <- dar(data=data[rowdata$TSS==TRUE,], coldata)
    res_nontss <- dar(data=data[rowdata$TSS==FALSE,], coldata)

    #combine TSS and nonTSS DAR results
    res <- rbind(res_tss, res_nontss) %>% mutate(BAFdep=case_when(change == "down" ~ TRUE, TRUE ~ FALSE)) #add column that binarizes peaks into BAF-dependent (significant accessibility loss) and independent
    #reorder rowdata
    rowdata <- rbind(rowdata[rowdata$TSS==TRUE,], rowdata[rowdata$TSS==FALSE,])
    return(list("results"=res, "row_meta"=rowdata))
}

In [619]:
#Run DAR analysis for each BAFinh timepoint

dar_IFN_1h <- dar_standard(data=counts_1h[,col_meta_1h$Condition %in% c("Veh", "IFN")], coldata=col_meta_1h[col_meta_1h$Condition %in% c("Veh", "IFN"),])#, rowdata=row_meta)
dar_Dex_1h <- dar_standard(data=counts_1h[,col_meta_1h$Condition %in% c("Veh", "Dex")], coldata=col_meta_1h[col_meta_1h$Condition %in% c("Veh", "Dex"),])#, rowdata=row_meta)

dar_IFN_6h <- dar_standard(data=counts_6h[,col_meta_6h$Condition %in% c("Veh", "IFN")], coldata=col_meta_6h[col_meta_6h$Condition %in% c("Veh", "IFN"),])#, rowdata=row_meta)
dar_Dex_6h <- dar_standard(data=counts_6h[,col_meta_6h$Condition %in% c("Veh", "Dex")], coldata=col_meta_6h[col_meta_6h$Condition %in% c("Veh", "Dex"),])#, rowdata=row_meta)

dar_BRM_IFN_1h <- dar_splitTSS(data=counts_1h[,col_meta_1h$Condition %in% c("IFN", "BRM_IFN")], coldata=col_meta_1h[col_meta_1h$Condition %in% c("IFN", "BRM_IFN"),], rowdata=row_meta)
dar_BRM_Dex_1h <- dar_splitTSS(data=counts_1h[,col_meta_1h$Condition %in% c("Dex", "BRM_Dex")], coldata=col_meta_1h[col_meta_1h$Condition %in% c("Dex", "BRM_Dex"),], rowdata=row_meta)

dar_BRM_IFN_6h <- dar_splitTSS(data=counts_6h[,col_meta_6h$Condition %in% c("IFN", "BRM_IFN")], coldata=col_meta_6h[col_meta_6h$Condition %in% c("IFN", "BRM_IFN"),], rowdata=row_meta)
dar_BRM_Dex_6h <- dar_splitTSS(data=counts_6h[,col_meta_6h$Condition %in% c("Dex", "BRM_Dex")], coldata=col_meta_6h[col_meta_6h$Condition %in% c("Dex", "BRM_Dex"),], rowdata=row_meta)

In [5]:
dar_BRM_1h <- dar_splitTSS(data=counts_1h[,col_meta_1h$Condition %in% c("Veh", "BRM")], coldata=col_meta_1h[col_meta_1h$Condition %in% c("Veh", "BRM"),], rowdata=row_meta)
dar_BRM_6h <- dar_splitTSS(data=counts_6h[,col_meta_6h$Condition %in% c("Veh", "BRM")], coldata=col_meta_6h[col_meta_6h$Condition %in% c("Veh", "BRM"),], rowdata=row_meta)


In [6]:
#write.table(cbind(row_meta[,1:3], dar_IFN_1h), "DAR_IFN_1h.tsv", sep='\t', row.names=FALSE)
#write.table(cbind(row_meta[,1:3], dar_IFN_6h), "DAR_IFN_6h.tsv", sep='\t', row.names=FALSE)
#write.table(cbind(row_meta[,1:3], dar_Dex_1h), "DAR_Dex_1h.tsv", sep='\t', row.names=FALSE)
#write.table(cbind(row_meta[,1:3], dar_Dex_6h), "DAR_Dex_6h.tsv", sep='\t', row.names=FALSE)

#write.table(cbind(dar_BRM_IFN_1h$row_meta[,1:3], dar_BRM_IFN_1h$results), "DAR_BRM_IFN_1h.tsv", sep='\t', row.names=FALSE)
#write.table(cbind(dar_BRM_IFN_6h$row_meta[,1:3], dar_BRM_IFN_6h$results), "DAR_BRM_IFN_6h.tsv", sep='\t', row.names=FALSE)
#write.table(cbind(dar_BRM_Dex_1h$row_meta[,1:3], dar_BRM_Dex_1h$results), "DAR_BRM_Dex_1h.tsv", sep='\t', row.names=FALSE)
#write.table(cbind(dar_BRM_Dex_6h$row_meta[,1:3], dar_BRM_Dex_6h$results), "DAR_BRM_Dex_6h.tsv", sep='\t', row.names=FALSE)

write.table(cbind(dar_BRM_1h$row_meta[,1:3], dar_BRM_1h$results), "DAR_BRM_1h.tsv", sep='\t', row.names=FALSE)
write.table(cbind(dar_BRM_6h$row_meta[,1:3], dar_BRM_6h$results), "DAR_BRM_6h.tsv", sep='\t', row.names=FALSE)